In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path


import statistics as s
from math import isnan
from itertools import filterfalse

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os, plotly.io as pio

import scipy.stats as scipy


In [ ]:
#define global variables
##scratch directory 
##work directory
##work1 ; directory for file from past experiment

data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-05_viral_disease_cohort"
results = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-11-05_viral_disease_cohort_creation" 
scratch = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/scratch/2025-11-05_viral_disease_cohort_creation"

#!mkdir {scratch}

In [ ]:
def get_cohort_dict_pkl():

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{data}/viral_disease_cohort_summary_stats_dataset_dict.pkl')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
#1 viral disease cohort overlap


In [ ]:
viral_overlap = dict()
    
def viral_overlap_dict(viral_cohort_dict):
    
    
    for key, table in viral_cohort_dict.items():
        
        person_IDs = table['person_id']
        
        concept_names_series = table['standard_concept_name']
        concept_names_str = concept_names_series[0]
    
        viral_overlap[concept_names_str]= set(person_IDs)
        
        
    return viral_overlap
    

In [ ]:
def viral_overlap_count(viral_overlap):
    
    max_overlap = []
    all_concept_names = viral_overlap.keys() #a list of dictionary keys (concept_names)
    less_than_20 = set()
    
    
    for concept_name in all_concept_names: #concept_name = concept name per row for each item in all_cocnept_names list
        max_concept_name = None
        overlap = 0
        person_ids = viral_overlap[concept_name] #list of person_id asscoiated with each key in viral_overlap dict
        person_ids_count = len(person_ids) #count of person_id asscociated with each key

        
        #initialize in case no overlap is found
        max_concept_count = 0
        overlap_percent = 0
        max_concept_overlap_percent = 0
        average = 0
        
        
        
        for concept_name_to_check in all_concept_names: #all concept names =  all the concept names again for each concept name indivdually in a list
                                                        #concept_name_to_check = each of those concept names individually from the list of concept_names used to check for each original key 

            person_ids_to_check = viral_overlap[concept_name_to_check] #person_ids associated with each concept_name , in the list of concept names to check for each original key
            number_of_overlaps = len(person_ids.intersection(person_ids_to_check)) #the overlap of those person ids, length of list of intersection of two list
            

        
            
            if person_ids == person_ids_to_check: #skipping self comparison (logical error)
                continue
            
            
        
            if number_of_overlaps > overlap:  #number_of_overlaps = if greater than 0, then overlap now = # of overlaps #upates the variable 
               
                overlap = number_of_overlaps #number of overlaps between each concept name where the # of overlaps is the greatest
                max_concept_name = concept_name_to_check #concep_name where the # of overlaps is the greatest 
            
                overlap_percent = int((overlap/person_ids_count)*100)  # (number of max_overlap/ person_id count )*100
                max_concept_count = len(viral_overlap[max_concept_name]) #count of person_id in the max_concept_name
                max_concept_overlap_percent = int((overlap/max_concept_count)*100) # (number of max_overlap/ max_concept person_id count )*100
                average = int((overlap_percent + max_concept_overlap_percent)/2) # (overlap_percent + max_concept_overlap_percent) / 2
      
                
        max_overlap.append([concept_name, person_ids_count, max_concept_name, max_concept_count, overlap, overlap_percent, max_concept_overlap_percent, average])
       
        
        final_df = pd.DataFrame(max_overlap)
        final_df.columns = ['standard_concept_name', 'count', 'max_concept_name', 'max_concept_count', 'overlap_count', 'overlap_percent', 'max_concept_overlap_percent', 'average']
        
    
    #add concept_ids
    
    final_cohort_concept_id = pd.read_csv(f"{data}/Final_cohort_query.csv")
    
    concept_id = list(final_cohort_concept_id['condition_concept_id'])
    
    final_df['condition_concept_id'] = concept_id
    
    final_df = final_df[['condition_concept_id', 'standard_concept_name', 'count', 'max_concept_name',
    'max_concept_count', 'overlap_count', 'overlap_percent',
    'max_concept_overlap_percent', 'average',]]
        
       

    return final_df


In [ ]:
#2 viral disease cohort summary stats


In [ ]:
def stats_summary_dict(df_dict):
   
    '''This function is to create a dataframe of the stats necessary for further descriptive statitics) '''

    stats_dict = {}

    for cohort, table in df_dict.items(): #need access to key also, to store results later in the loop / tuple

        
        #Calculate Age ---

        # Ensure date columns are datetime objects, handling potential errors
        # Using .loc to avoid a potential SettingWithCopyWarning
        table.loc[:, 'first_diag_date'] = pd.to_datetime(table['first_diag_date'], errors="coerce")
        table.loc[:, 'date_of_birth'] = pd.to_datetime(table['date_of_birth'], errors="coerce")

        # Calculate age in years
        # This will result in NaN if either date was invalid (became NaT)
        table['age'] = (table['first_diag_date'] - table['date_of_birth']).dt.days / 365.25

        
        # Strip NONE and replace with NaN values
        table.replace("NONE", np.nan, inplace=True)
        
    
        #concept_name = no_dups['standard_concept_name']
        #sex = table['sex_at_birth']
        #socio = table['zip_code']
        #race_stats = table['race']
        #self_reported = table['self_reported_category']
        #ethnicity = table['updated_race']
        age_stats = table['age']
        concept_visit_stats = table['visits_for_concept']
        gen_diag_stats = table['concept_count_ehr']
        gen_visit_stats = table['visits_all_concepts']
        cohort_count = list(table['person_id'])
        
        # Strip NaN values
        age = list(filterfalse(isnan, age_stats))
        concept_visit = list(filterfalse(isnan, concept_visit_stats))   
        gen_diag = list(filterfalse(isnan, gen_diag_stats))
        gen_visits = list(filterfalse(isnan, gen_visit_stats))
       
        
        #build dataframe of results
        df = pd.DataFrame ({
             
            'concept_id' : [cohort],
            'cohort count' : [len(cohort_count)],
            'age_median' : [s.median(age)],
            'age_mean' : [round(s.mean(age))],
            'age_min' : [min(age)],
            'age_max' : [max(age)],            
            'age_Q1' : [np.percentile(age, [25])],
            'age_Q3' : [np.percentile(age, [75])],
            
            
            'concept_visit_median' : [s.median(concept_visit)],
            'concept_visit_mean' : [s.mean(concept_visit)],
            'concept_visit_min' : [min(concept_visit)],
            'concept_visit_max' : [max(concept_visit)],            
            'concept_visit_Q1' : [np.percentile(concept_visit, [25])],
            'concept_visit_Q3' : [np.percentile(concept_visit, [75])],
            
            
            'total_diag_median' : [s.median(gen_diag)],
            'total_diag_mean' : [s.mean(gen_diag)],
            'total_diag_min' : [min(gen_diag)],
            'total_diag_max' : [max(gen_diag)],            
            'total_diag_Q1' : [np.percentile(gen_diag, [25])],
            'total_diag_Q3' : [np.percentile(gen_diag, [75])],
            
            
            'total_visits_median' : [s.median(gen_visits)],
            'total_visits_mean' : [s.mean(gen_visits)],
            'total_visits_min' : [min(gen_visits)],
            'total_visits_max' : [max(gen_visits)],            
            'total_visits_Q1' : [np.percentile(gen_visits, [25])],
            'total_visits_Q3' : [np.percentile(gen_visits, [75])]
     
        })
    
        #wrap results in a dictionary
        stats_dict[cohort] = df
     
    return stats_dict

In [ ]:

def make_stats_table_figure(df, title):
    
    # transpose & reset index
    df_t = df.T.reset_index()
    df_t.columns = ["Descriptive stat", "Value"]

    # build the table figure
    fig = go.Figure(
        go.Table(
            header=dict(
                values=df_t.columns,
                height = 30,
                fill_color="black",
                font=dict(color="white", size=12),
                align="left"
            ),
            cells=dict(
                values=[df_t["Descriptive stat"], df_t["Value"]],
                height = 25,
                fill_color=["lightblue", "lightgray"],
                align="left"
            )
        )
    )
    fig.update_layout(
        title= "Viral disease cohort descriptive statatistics",
        height=300,
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig



In [ ]:

def make_plotly_stats_table_html(table_dict):

    # 1) Make sure results/ exists
    output_dir = f'{data}' #folder
    os.makedirs(output_dir, exist_ok=True)

    # 2) Generate your dict of Figures
    df = table_dict
    

    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral disease cohort descriptive stats table</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    
    
    output_path = f"{data}/cohorts_summary_plotly_tables.html" #output path, folder to save and html file name
    with open(output_path, "w", encoding="utf-8") as f:
        
        f.write(html_header) #writing the header 

        first = True  #Sets a flag to track whether this is the first figure being written. It ensures that Plotly JS is only included once.
        
        for cohort, figure in df.items():
            f.write(f"<h2>Cohort: {cohort}</h2>\n") #write the image header, notated in html_header (ie) like metadata)
          
            #convert plotly to an html snippet
            snippet = pio.to_html(
                figure,
                full_html=False,
                include_plotlyjs="inline" if first else False  # Ensures the Plotly JavaScript is only embedded once (for the first figure) to avoid bloating the file.
            )
            
            first = False #After writing the first figure, first is set to False so that Plotly JS isn’t redundantly included for subsequent figures.

  
            f.write(snippet + "<hr>\n") #adding plot

        f.write(html_footer) #adding footer

    print(f"✅ Wrote {output_path}") #completion


In [ ]:
def get_cohort_updated_race(df):
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 1. Merge Race/Ethnicity ---
    
    # Define the helper function for merging
    def merge_race_ethnicity_data(row):
        """Helper function to apply row-wise."""
        if row["ethnicity"] == "Hispanic or Latino":
            return row["ethnicity"]
        else:
            return row["race"]

    # Apply the function to create the 'updated_race' column
    wrangled_df["updated_race"] = wrangled_df.apply(merge_race_ethnicity_data, axis=1)

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]


    return wrangled_df


In [ ]:
def stats_summary_plots(df_dict):
   
    '''This function is to create distribution plots from the data from the merge_data_table() function'''

    figs_dict = {}
    
    
    for cohort, table in df_dict.items(): #need access to key also, to store results later in the loop / tuple
        
        
            
        #create series of data    
        sex = table['sex_at_birth']
        ethnicity = table['updated_race']
        socio = table['zip_code']
        age_stats = table['age']
        concept_visit_stats = table['visits_for_concept'] 
        gen_diag_stats = table['concept_count_ehr']
        gen_visit_stats = table['visits_all_concepts']
        
        
        
        #create plots
        fig = make_subplots(rows = 2, cols = 4, vertical_spacing= 0.30, horizontal_spacing = .1)
          
        trace1 = go.Histogram(x = sex, name = 'sex')
        trace2 = go.Histogram(x = age_stats, name = 'age', xbins = dict(size = '1'), autobinx = False)
        trace3 = go.Histogram(x = ethnicity, name = 'ethnicity') 
        trace4 = go.Histogram(x = socio, name = 'zip code', xbins = dict(size = '1'), autobinx = False)
        trace5 = go.Histogram(x = concept_visit_stats, name = 'concept visit count', xbins = dict(size = '1'), autobinx = False)
        trace6 = go.Histogram(x = gen_diag_stats, name = 'total diagnosis count', xbins = dict(size = '1'), autobinx = False)
        trace7 = go.Histogram(x = gen_visit_stats, name = 'total visits', xbins = dict(size = '1'), autobinx = False)
        

            
        fig.add_trace(trace1, 1, 1)
        fig.add_trace(trace2, 1, 2)
        fig.add_trace(trace3, 1, 3)
        fig.add_trace(trace4, 1, 4)
        
        fig.add_trace(trace5, 2, 1)
        fig.add_trace(trace6, 2, 2)
        fig.add_trace(trace7, 2, 3)
  
       
        
        
        #update subplot axes
        #row 1
        fig.update_xaxes(title_text="sex", tickangle= -45, row = 1, col = 1)
        fig.update_yaxes(title_text="patient count", row = 1, col = 1)
        
        fig.update_xaxes(title_text="age", row = 1, col = 2)
        fig.update_yaxes(title_text="patient count", row = 1, col = 2)
        
        fig.update_xaxes(title_text='ethnicity', tickangle= -45, row = 1, col = 3)
        fig.update_yaxes(title_text="patient count", row = 1, col = 3)
        
        
        fig.update_xaxes(title_text="zip code", tickangle= -45, row = 1, col = 4)
        fig.update_yaxes(title_text="patient count", row = 1, col = 4)
        
        
        
        #row2
        fig.update_xaxes(title_text="unique concept visit count", row = 2, col = 1)
        fig.update_yaxes(title_text="patient count", row = 2, col = 1)
        
        fig.update_xaxes(title_text="diagnosis count", row = 2, col = 2)
        fig.update_yaxes(title_text="patient count", row = 2, col = 2)
        
        fig.update_xaxes(title_text="visits count", row = 2, col = 3)
        fig.update_yaxes(title_text="patient count", row = 2, col = 3)
        
        
        
        fig.update_xaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )
        fig.update_yaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )

        
            
        fig.update_layout(title_text= F'Patient count distribution: {cohort}',legend_title_text = "Variable", 
                            plot_bgcolor='white', bargap = 0.2, height = 700, width = 1000, margin=dict(l=80, r=80, t=80, b=80))          

        figs_dict[cohort] = fig
        
    return figs_dict

In [ ]:
def create_cohort_summary_plots_html():


    # 2) Generate your dict of Figures
    figs_dict = stats_summary_plots(wrangled_dict)

    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral Disease Cohort Summary Statistics</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    output_path = f"{data}/cohort_summary_plots.html"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_header)

        first = True
        for cohort_id, fig in figs_dict.items():
            f.write(f"<h2>Cohort: {cohort_id}</h2>\n")

            snippet = pio.to_html(
                fig,
                full_html=False,
                include_plotlyjs="inline" if first else False
            )
            first = False

            f.write(snippet + "\n<hr>\n")

        f.write(html_footer)

    print(f"✅ Wrote {output_path}")




In [ ]:
def diagnosis_date_freq(diagnosis_dates_df):
    
    """
    For each cohort in diagnosis_dates_df, plots histograms of
    first & last diagnosis dates, and saves each figure to a dictionary, with its respective concept_id key
    
    """
    
  
    date_dict = {}
    
    for cohort, table in diagnosis_dates_df.items():
            
              
            first = table['first_diag_date']
            last = table['last_diag_date']
    
            
            fig = make_subplots(rows = 1, cols = 2)
          

            
            first_trace = go.Histogram(x = first, name = 'first diagnosis', xbins = dict(
                                        size='M1'),autobinx = False)
            
            last_trace = go.Histogram(x = last, name = 'last diagnosis', xbins = dict(
                                     size='M1'), autobinx = False)
            
            
            fig.add_trace(first_trace, 1, 1)
            fig.add_trace(last_trace, 1, 2)
            
            
            
            fig.update_xaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )
            fig.update_yaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )
            
            fig.update_layout(title_text= F'First and Last diagnosis distribution: {cohort}', 
                xaxis_title_text='Diagnosis Date', 
                yaxis_title_text='Patient Count', 
                bargap = 0.2)
            
            
            date_dict[cohort] = fig
            
    return date_dict

In [ ]:
def create_cohort_date_plots_html():

    # 1) Make sure results/ exists
    output_dir = data
    os.makedirs(output_dir, exist_ok=True)

    # 2) Generate your dict of Figures
    dates_dict = diagnosis_date_freq(cohort_dict)


    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral Disease Cohort Date distributions</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    output_path = f"{data}/all_cohorts_dates_plot.html"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_header)

        first = True
        for cohort_id, fig in dates_dict.items():
            f.write(f"<h2>Cohort: {cohort_id}</h2>\n")

            snippet = pio.to_html(
                fig,
                full_html=False,
                include_plotlyjs="inline" if first else False
            )
            first = False

            f.write(snippet + "\n<hr>\n")

        f.write(html_footer)

    print(f"✅ Wrote {output_path}")

In [ ]:
#post cohort summary stats analysis

'''
1. identify outliers view skew test
    age, concept visit count, total visit count, and total diagnosis count.
    
2. Sex : identify any 70/30 proportion threshold within the chorts for sex bias
3. Race, zip code , find highest peak, and find what proportion of the cohort is found there 

input: df1 - original cohort dataframe dictionary

1; array of values from df1 table, similar to code used for the stats summary table
2: count of either men or women, and calculate the proportion to see if any meet the 70/30 threshhold
3. sum the count of each unique x value, take the highest count and calculate the proportion, series of these resuslts 

output: go.table with results
'''

In [ ]:

def post_cohort_analysis(df_dict): 
    
   
    post_stats_dict = {}

    for cohort, table in df_dict.items(): #need access to key also, to store results later in the loop / tuple


        
        # Strip NONE and replace with NaN values
        table.replace("NONE", np.nan, inplace=True)
        
    
        #concept_name = no_dups['standard_concept_name']
        sex = (table['sex_at_birth']).astype('category')
        socio = table['zip_code'].astype('category')
        ethnicity = table['updated_race'].astype('category')
        age_stats = table['age']
        concept_visit_stats = table['visits_for_concept']
        gen_diag_stats = table['concept_count_ehr']
        gen_visit_stats = table['visits_all_concepts']
        cohort_count = list(table['person_id'])
        
        # Strip NaN values
        age = list(filterfalse(isnan, age_stats))
        concept_visit = list(filterfalse(isnan, concept_visit_stats))   
        gen_diag = list(filterfalse(isnan, gen_diag_stats))
        gen_visits = list(filterfalse(isnan, gen_visit_stats))
       
        
        
        # 1. identify outliers via skew test
    
        age_skew = round(scipy.skew(age), 3)
        concept_visit_skew = round(scipy.skew(concept_visit), 3)
        total_diag_skew = round(scipy.skew(gen_diag), 3)
        total_visit_skew = round(scipy.skew(gen_visits), 3)
        
        
        
        
        # 2. Sex : identify any 70/30 proportion threshold within the chorts for sex bias
        #use female as proxy
     
        male = round(sex.isin(['Male']).sum() / len(cohort_count) * 100, 2)
        female = round(sex.isin(['Female']).sum() / len(cohort_count) * 100, 2)
 
        
        
        # 3. Race, zip code , find highest peak, and find what proportion of the cohort is found there 

        max_ethnicity = ethnicity.value_counts().max()  #max race value
        which_ethnicity = ethnicity.value_counts().idxmax()
     
        max_zip = socio.value_counts().max()
        which_zip = socio.value_counts().idxmax()
        
        max_ethnicity_prop = round(max_ethnicity / len(cohort_count) * 100, 2)
        max_zip_prop = round(max_zip / len(cohort_count) * 100, 2)
        
       
               
        #build dataframe of results
        df = pd.DataFrame ({ 
            
            'concept_id' : [cohort], 
            'male_prop' : [male],
            'female_prop' : [female],
            'max_ethnicity' : [which_ethnicity],
            'max_ethnicity_prop' : [max_ethnicity_prop],
            'max_zip' : [which_zip],
            'max_zip_prop' : [max_zip_prop],
            'skew_of_age' : [age_skew],
            'skew_of_concept_visits' : [concept_visit_skew],
            'skew_of_total_diag' : [total_diag_skew],            
            'skew_of_total_visits' : [total_visit_skew],

               })
        
           #wrap results in a dictionary
        post_stats_dict[cohort] = df
     
    return post_stats_dict

In [ ]:
def post_summary_plotly_table_html(post_table_dict):

    # 1) Make sure results/ exists
    output_dir = 'post_cohort_summary_table' #folder
    os.makedirs(output_dir, exist_ok=True)

    # 2) Generate your dict of Figures
    df = post_table_dict
    

    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral disease cohort post descriptive stats table</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    
    
    output_path = f"{data}/post_cohorts_summary_plotly_tables.html" #output path, folder to save and html file name
    with open(output_path, "w", encoding="utf-8") as f:
        
        f.write(html_header) #writing the header 

        first = True  #Sets a flag to track whether this is the first figure being written. It ensures that Plotly JS is only included once.
        
        for cohort, figure in df.items():
            f.write(f"<h2>Cohort: {cohort}</h2>\n") #write the image header, notated in html_header (ie) like metadata)
          
            #convert plotly to an html snippet
            snippet = pio.to_html(
                figure,
                full_html=False,
                include_plotlyjs="inline" if first else False  # Ensures the Plotly JavaScript is only embedded once (for the first figure) to avoid bloating the file.
            )
            
            first = False #After writing the first figure, first is set to False so that Plotly JS isn’t redundantly included for subsequent figures.

  
            f.write(snippet + "<hr>\n") #adding plot

        f.write(html_footer) #adding footer

    print(f"✅ Wrote {output_path}") #completion

In [ ]:
#Main
#cell to call all functions


##overlap
#cohort_dict =  get_cohort_dict_pkl()

#overlap_dict = viral_overlap_dict(cohort_dict)

#overlap = viral_overlap_count(overlap_dict)


##stats


#stats = stats_summary_dict(cohort_dict)

#loop through stats table results and apply the function to build the table
#fig_dict = {}
#for name, df in stats.items():
#    fig_dict[name] = make_stats_table_figure(df, title=name)


#make_plotly_stats_table_html(fig_dict)

#wrangled_dict = dict()
#for name, df in cohort_dict.items():
    
#    updated = get_cohort_updated_race(df)
    
#    wrangled_dict[name] = updated


#create_cohort_summary_plots_html()

#create_cohort_date_plots_html()


##post stats

#post_stats = post_cohort_analysis(wrangled_dict)

#post_summary_dict = {}
#loop through stats table results and apply the function to build the table
#for name, df in post_stats.items():
#    post_summary_dict[name] = make_stats_table_figure(df, title=name) #used previous function created for the stats summary

#post_summary_plotly_table_html(post_summary_dict)
    

In [ ]:
#save to csv

#overlap.to_csv(f"{scratch}/viral_cohort_person_id_overlap.csv")

In [ ]:
#visualization cell

#cohort_dict

#overlap_dict

#overlap

#stats

#fig_dict

#post_stats

#post_summary_dict